In [1]:
#セル1 --- ライブラリのインポート ---

import matplotlib
matplotlib.use('Agg')  

import tkinter as tk
from tkinter import filedialog
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
import re
# ★ 修正：sosfiltfilt を追加しました ★
from scipy.signal import butter, filtfilt, sosfiltfilt
from collections import defaultdict
import openpyxl
from openpyxl.drawing.image import Image as OpenpyxlImage
from openpyxl.utils import get_column_letter
from io import BytesIO

print("セル1: ライブラリのインポートが完了しました。")

セル1: ライブラリのインポートが完了しました。


In [2]:
# セル2--- 設定セクション ---

# ファイル読み込み設定
ROWS_TO_SKIP = 70
FILE_ENCODING = 'cp932'

# 解析する時間範囲を秒で指定（行数はファイルごとのサンプリング周期から換算する）
START_TIME_SEC = 1.0     # 解析開始時間 [s]
END_TIME_SEC   = None    # 解析終了時間 [s]（None = ファイルの最後まで）

# 処理対象の列
TARGET_COLUMNS = ['(1)HA-V05']

# 抽出パラメータの設定
TIME_BEFORE_MAX_SEC = 0.7  # 最高値から何秒前のデータを取得するか
RISE_THRESHOLD_V = 2.0     # ★注意: オフセットがないため、実際の絶対電圧(V)に合わせて調整してください★

# データ処理設定
FILTER_CUTOFF_HZ = 300
FILTER_ORDER = 5

print("セル2: 設定の読み込みが完了しました。")

セル2: 設定の読み込みが完了しました。


In [3]:
#セル3 --- 加工関数の定義 ---

def generate_plot_image(df, points_data, original_filepath, output_path, actual_sampling_rate_hz, fig_width=10, fig_height=4, fig_dpi=150, x_lim=None, y_lim=None, plot_style=1):
    base_filename = os.path.basename(original_filepath)
    
    fig, axes = plt.subplots(len(TARGET_COLUMNS), 1, figsize=(fig_width, fig_height * len(TARGET_COLUMNS)))
    
    if len(TARGET_COLUMNS) == 1: axes = [axes]

    for i, col in enumerate(TARGET_COLUMNS):
        ax = axes[i]
        smooth_col = f'{col}_smooth'
        
        # --- ★ セル6から指示されたスタイル(plot_style)で描き分ける ★ ---
        if plot_style == 2:
            # スタイル2：透かしバージョン（元の波形を濃く、フィルターを半透明にする）
            ax.plot(df['Time (s)'], df[col], label='Original', color='gray', alpha=0.9, linewidth=1.5)
            ax.plot(df['Time (s)'], df[smooth_col], label='Smoothed', color='blue', alpha=0.5, linewidth=2.0)
        else:
            # スタイル1：従来バージョン（元の波形は薄く、フィルターをくっきりと）
            ax.plot(df['Time (s)'], df[col], label='Original', color='lightgray', alpha=0.7, linewidth=1.0)
            ax.plot(df['Time (s)'], df[smooth_col], label='Smoothed', color='blue', alpha=1.0, linewidth=1.5)
        # -----------------------------------------------------------------

        pts = points_data.get(col, {})
        
        if pd.notna(pts.get('max_time')):
            ax.plot(pts['max_time'], pts['max_val'], marker='*', markeredgecolor='red', markerfacecolor='none', markersize=16, markeredgewidth=1.5, label='Max Point')
            
        if pd.notna(pts.get('bef_time')):
            ax.plot(pts['bef_time'], pts['bef_val'], marker='*', markeredgecolor='cyan', markerfacecolor='none', markersize=16, markeredgewidth=1.5, label=f'{TIME_BEFORE_MAX_SEC}s Before Max')
            
        if pd.notna(pts.get('rise_time')):
            ax.plot(pts['rise_time'], pts['rise_val'], marker='*', markeredgecolor='green', markerfacecolor='none', markersize=16, markeredgewidth=1.5, label='Rise Point')
            ax.axhline(y=RISE_THRESHOLD_V, color='green', linestyle='--', alpha=0.5, label=f'Threshold ({RISE_THRESHOLD_V}V)')

        title_text = f"{base_filename} (Sampling: {int(actual_sampling_rate_hz)} Hz)"
        ax.set_title(title_text, fontsize=12, fontweight='bold')
        
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('Voltage (V)')
        ax.legend(fontsize='small', loc='upper right')
        ax.grid(True)
        
        if x_lim is not None:
            ax.set_xlim(x_lim)
        if y_lim is not None:
            ax.set_ylim(y_lim)

    fig.tight_layout()
    
    img_buffer = BytesIO()
    fig.savefig(img_buffer, format='png', dpi=fig_dpi)
    img_buffer.seek(0)
    
    fig.clear()
    plt.close(fig)
    return img_buffer


# ★ 引数に plot_style を追加しました
def process_file(filepath, make_graph=False, fig_width=10, fig_height=4, fig_dpi=150, x_lim=None, y_lim=None, plot_style=1):
    actual_sampling_rate_hz = 100000.0
    found_sampling_rate = False        
    
    try:
        with open(filepath, 'r', encoding=FILE_ENCODING, errors='ignore') as f:
            for _ in range(50):
                line = f.readline()
                if not line: break
                if 'サンプリング周期' in line:
                    match = re.search(r'(\d+)\s*μs', line)
                    if match:
                        us_val = float(match.group(1))
                        if us_val > 0:
                            actual_sampling_rate_hz = 1000000.0 / us_val
                            found_sampling_rate = True
                            break
    except Exception as e:
        print(f" -> [警告] ヘッダー読み取りエラー: {e}")

    try:
        nrows = None if END_TIME_SEC is None else int(END_TIME_SEC * actual_sampling_rate_hz) + 1  # END_TIME_SEC までの行数
        df = pd.read_csv(filepath, skiprows=ROWS_TO_SKIP, encoding=FILE_ENCODING, on_bad_lines='skip', nrows=nrows, low_memory=False)
        # ファイル末尾のマーク情報（#BeginMark・#EndMark の行）はデータではないので取り除く
        df = df[~df.iloc[:, 0].astype(str).str.startswith('#')].reset_index(drop=True)
    except Exception as e:
        print(f"ファイル '{os.path.basename(filepath)}' 読み込みエラー: {e}")
        return None, None

    try:
        for col in TARGET_COLUMNS:
            if col not in df.columns:
                print(f" -> [警告] 列 '{col}' が見つかりません。")
                df[col] = 0
            df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

        df['Time (s)'] = np.arange(0, len(df)) * (1.0 / actual_sampling_rate_hz)
        
        nyquist = 0.5 * actual_sampling_rate_hz
        sos = butter(FILTER_ORDER, FILTER_CUTOFF_HZ / nyquist, btype='low', analog=False, output='sos')
        
        for col in TARGET_COLUMNS:
            df[f'{col}_smooth'] = sosfiltfilt(sos, df[col])
            
        start_row = int(round(START_TIME_SEC * actual_sampling_rate_hz))
        df = df.iloc[start_row:].reset_index(drop=True)
            
    except Exception as e:
        print(f" -> データ前処理エラー: {e}")
        return None, None

    if df.empty:
        print(f" -> [警告] {START_TIME_SEC}秒以降のデータがありません。START_TIME_SEC を見直してください。")
        return None, None
    
    results_for_this_file = {'source_file': os.path.basename(filepath)}
    points_data_for_plot = {}
    
    for col in TARGET_COLUMNS:
        smooth_col = f'{col}_smooth'
        
        max_idx = df[smooth_col].idxmax()
        max_val = df.loc[max_idx, smooth_col]
        max_time = df.loc[max_idx, 'Time (s)']
        
        target_time_before = max_time - TIME_BEFORE_MAX_SEC
        valid_rows = df[df['Time (s)'] <= target_time_before]
        
        if valid_rows.empty:
            val_before = np.nan
            actual_time_before = np.nan 
        else:
            closest_smaller_idx = valid_rows.index[-1]
            val_before = df.loc[closest_smaller_idx, smooth_col]
            actual_time_before = df.loc[closest_smaller_idx, 'Time (s)']
            
        rise_candidates = df[df[smooth_col] >= RISE_THRESHOLD_V]
        if not rise_candidates.empty:
            rise_idx = rise_candidates.index[0]
            rise_time = df.loc[rise_idx, 'Time (s)']
            rise_val = df.loc[rise_idx, smooth_col]
        else:
            rise_time = np.nan
            rise_val = np.nan
            
        results_for_this_file[f'{col}_最高値(V)'] = max_val
        results_for_this_file[f'{col}_最高値の時間(s)'] = max_time
        results_for_this_file[f'{col}_{TIME_BEFORE_MAX_SEC}秒前の値(V)'] = val_before
        results_for_this_file[f'{col}_{TIME_BEFORE_MAX_SEC}秒前の時間(s)'] = actual_time_before
        results_for_this_file[f'{col}_立ち上がり値(V)'] = rise_val
        results_for_this_file[f'{col}_立ち上がり時間(s)'] = rise_time
        
        points_data_for_plot[col] = {
            'max_time': max_time, 'max_val': max_val,
            'bef_time': actual_time_before, 'bef_val': val_before,
            'rise_time': rise_time, 'rise_val': rise_val
        }

    image_buffer = None
    if make_graph:
        # ★ セル6から指示された plot_style を画像生成関数に渡す
        image_buffer = generate_plot_image(df, points_data_for_plot, filepath, None, actual_sampling_rate_hz, fig_width, fig_height, fig_dpi, x_lim, y_lim, plot_style)

    return results_for_this_file, image_buffer

print("セル3: 関数の定義が完了しました。")

セル3: 関数の定義が完了しました。


In [4]:
#セル4---加工対象のメインフォルダの指定---

print("フォルダ選択ダイアログを開いています...（画面の裏側に隠れている場合があります）")

root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

folder_path = filedialog.askdirectory(title="解析したいCSVファイルが入っている【メインフォルダ】を選択してください")

if not folder_path:
    print("キャンセルされました。")
elif not os.path.isdir(folder_path):
    print("エラー: パスが見つかりません。")
else:
    print(f"選択されたフォルダ: {folder_path}")
    
    search_pattern = os.path.join(folder_path, '**', '*.csv')
    csv_files = glob.glob(search_pattern, recursive=True)
    # このノートブックやデータ圧縮コードが出力したCSVは、入力から除外する
    def is_output_csv(path):
        name = os.path.basename(path)
        return name.startswith(('summary_', 'max_summary_')) or name.endswith('_smoothed_10k.csv')
    csv_files = [f for f in csv_files if not is_output_csv(f)]
    
    if not csv_files:
        print("エラー: CSVファイルが見つかりませんでした。")
    else:
        print(f"{len(csv_files)}個のCSVファイルを検出しました。")
        files_by_subdir = defaultdict(list)
        for f in csv_files:
            relative_path = os.path.relpath(os.path.dirname(f), folder_path)
            if relative_path == '.': relative_path = "（メインフォルダ直下）"
            files_by_subdir[relative_path].append(f)
            
        print("セル4: 準備完了です。")

フォルダ選択ダイアログを開いています...（画面の裏側に隠れている場合があります）
選択されたフォルダ: C:/Users/0uh2j/Desktop/実験データ2026/01_本実験データ/03_GPPS実験加工処理データ/温度実験/NR_T/csv/5data
360個のCSVファイルを検出しました。
セル4: 準備完了です。


In [5]:
#セル5---数値データの抽出とCSVファイルへの保存---

if not 'csv_files' in locals() or not csv_files:
    print("エラー: セル4を先に実行してください。")
else:
    print("\nデータの抽出を開始します（グラフ描画なし・高速）...")
    all_results = []
    
    for subdir, files in sorted(files_by_subdir.items()):
        print(f"\n--- サブフォルダ: {subdir} ---")
        for filepath in sorted(files):
            print(f"抽出中: {os.path.basename(filepath)}")
            
            results, _ = process_file(filepath, make_graph=False)
            if results: all_results.append(results)

    if all_results:
        output_summary_path = os.path.join(folder_path, 'max_summary_all_features.csv')
        summary_df = pd.DataFrame.from_records(all_results)
        
        summary_df.rename(columns={'source_file': '元ファイル'}, inplace=True)

        base_names = summary_df['元ファイル'].str.replace(r'\.csv$', '', case=False, regex=True)
        parts = base_names.str.split('_', expand=True)
        
        # 4つのブロックを安全に取得（0:番号, 1:温度, 2:速度, 3:厚み）
        file_num_col = parts[0] if 0 in parts.columns else pd.Series(index=summary_df.index, dtype=str)
        temp_col = parts[1] if 1 in parts.columns else pd.Series(index=summary_df.index, dtype=str)
        speed_col = parts[2] if 2 in parts.columns else pd.Series(index=summary_df.index, dtype=str)
        thickness_col = parts[3] if 3 in parts.columns else pd.Series(index=summary_df.index, dtype=str)
        
        # ★ 全角半角の変換処理を削除し、不要な文字（℃、mm）を取り除いてシンプルに数値化 ★
        summary_df['ファイル番号'] = pd.to_numeric(file_num_col, errors='coerce')
        summary_df['加熱シリンダ温度[℃]'] = pd.to_numeric(temp_col.fillna('').astype(str).str.replace('℃', '', case=False), errors='coerce')
        summary_df['射出速度[mm/s]'] = pd.to_numeric(speed_col, errors='coerce')
        summary_df['キャビティ厚み[mm]'] = pd.to_numeric(thickness_col.fillna('').astype(str).str.replace('mm', '', case=False), errors='coerce')
        
        # ご要望の4段階ルールで完璧に並び替える
        summary_df = summary_df.sort_values(
            by=['キャビティ厚み[mm]', '加熱シリンダ温度[℃]', '射出速度[mm/s]', 'ファイル番号']
        )
        
        # CSVの列の並び順を整理
        final_column_order = ['元ファイル', 'キャビティ厚み[mm]', '加熱シリンダ温度[℃]', '射出速度[mm/s]']
        for col in TARGET_COLUMNS:
            final_column_order.extend([
                f'{col}_最高値(V)', f'{col}_最高値の時間(s)',
                f'{col}_{TIME_BEFORE_MAX_SEC}秒前の値(V)', f'{col}_{TIME_BEFORE_MAX_SEC}秒前の時間(s)',
                f'{col}_立ち上がり値(V)', f'{col}_立ち上がり時間(s)'
            ])
            
        summary_df_final = summary_df[final_column_order]
        summary_df_final.to_csv(output_summary_path, index=False, encoding='utf-8-sig')
        print(f"\n★ CSV抽出完了: '{output_summary_path}'")


データの抽出を開始します（グラフ描画なし・高速）...

--- サブフォルダ: 1.0mmNR\200℃ ---
抽出中: 001_200℃_010_1.0mmNR.csv
抽出中: 002_200℃_010_1.0mmNR.csv
抽出中: 003_200℃_010_1.0mmNR.csv
抽出中: 004_200℃_010_1.0mmNR.csv
抽出中: 005_200℃_010_1.0mmNR.csv
抽出中: 006_200℃_020_1.0mmNR.csv
抽出中: 007_200℃_020_1.0mmNR.csv
抽出中: 008_200℃_020_1.0mmNR.csv
抽出中: 009_200℃_020_1.0mmNR.csv
抽出中: 010_200℃_020_1.0mmNR.csv
抽出中: 011_200℃_040_1.0mmNR.csv
抽出中: 012_200℃_040_1.0mmNR.csv
抽出中: 013_200℃_040_1.0mmNR.csv
抽出中: 014_200℃_040_1.0mmNR.csv
抽出中: 015_200℃_040_1.0mmNR.csv
抽出中: 016_200℃_080_1.0mmNR.csv
抽出中: 017_200℃_080_1.0mmNR.csv
抽出中: 018_200℃_080_1.0mmNR.csv
抽出中: 019_200℃_080_1.0mmNR.csv
抽出中: 020_200℃_080_1.0mmNR.csv
抽出中: 021_200℃_120_1.0mmNR.csv
抽出中: 022_200℃_120_1.0mmNR.csv
抽出中: 023_200℃_120_1.0mmNR.csv
抽出中: 024_200℃_120_1.0mmNR.csv
抽出中: 025_200℃_120_1.0mmNR.csv
抽出中: 026_200℃_160_1.0mmNR.csv
抽出中: 027_200℃_160_1.0mmNR.csv
抽出中: 028_200℃_160_1.0mmNR.csv
抽出中: 029_200℃_160_1.0mmNR.csv
抽出中: 030_200℃_160_1.0mmNR.csv
抽出中: 031_200℃_200_1.0mmNR.csv
抽出中: 032_20

In [6]:
#セル6---（任意）グラフ画像の生成と、エクセルファイルへの一覧表示---

if not 'csv_files' in locals() or not csv_files:
    print("エラー: セル4を先に実行してください。")
else:
    # =========================================================
    # ★ エクセル出力レイアウト ＆ グラフ設定 ★
    # =========================================================
    GRAPH_WIDTH_INCH  = 10     
    GRAPH_HEIGHT_INCH = 4      
    GRAPH_DPI         = 150    
    SPACE_X           = 25     
    SPACE_Y           = 60     

    # ★ グラフの描画スタイル（見え方）の選択 ★
    # 1 = 従来版（青いフィルター波形を強調。元の波形は薄い灰色）
    # 2 = 透かし版（元の波形を濃い灰色にし、青い波形を半透明にして透けさせる）
    PLOT_STYLE = 1

    # グラフの軸の範囲を固定する設定
    X_AXIS_LIM = (0.0, 12.0)  
    Y_AXIS_LIM = (0.0, 6.0)  
    # =========================================================

    print("\nグラフの生成とExcelへの貼り付けを開始します（少し時間がかかります）...")
    all_images_by_subdir = defaultdict(list)
    
    def custom_sort_key(filepath):
        basename = os.path.basename(filepath).lower().replace('.csv', '')
        parts = basename.split('_')
        try:
            file_num = float(parts[0]) if len(parts) > 0 else 9999
            temp = float(parts[1].replace('℃', '')) if len(parts) > 1 else 9999
            speed = float(parts[2]) if len(parts) > 2 else 9999
            thickness = float(parts[3].replace('mm', '')) if len(parts) > 3 else 9999
            return (thickness, temp, speed, file_num)
        except ValueError:
            return (9999, 9999, 9999, 9999)

    for subdir, files in sorted(files_by_subdir.items()):
        print(f"\n--- サブフォルダ: {subdir} のグラフ生成 ---")
        for filepath in sorted(files, key=custom_sort_key):
            print(f"グラフ作成中: {os.path.basename(filepath)}")
            
            _, image_buffer = process_file(
                filepath, 
                make_graph=True, 
                fig_width=GRAPH_WIDTH_INCH, 
                fig_height=GRAPH_HEIGHT_INCH, 
                fig_dpi=GRAPH_DPI,
                x_lim=X_AXIS_LIM,  
                y_lim=Y_AXIS_LIM,
                plot_style=PLOT_STYLE
            )
            
            if image_buffer:
                all_images_by_subdir[subdir].append(image_buffer)
                
    if not all_images_by_subdir:
        print("グラフ画像を生成できませんでした。")
    else:
        print("\nすべてのグラフを1つのExcelファイルにまとめています...")
        
        # --- ★ 修正：選んだスタイルに合わせてファイル名を自動で変更する ★ ---
        if PLOT_STYLE == 2:
            style_name = "透かし版"
        else:
            style_name = "従来版"
            
        output_excel_path = os.path.join(folder_path, f'max_summary_plots_COMBINED_{style_name}.xlsx')
        # ----------------------------------------------------------------------
        
        wb = openpyxl.Workbook()
        ws = wb.active
        ws.title = "All_Graphs"
        ws.column_dimensions['A'].width = 30 
        
        current_row = 1
        max_graphs_in_a_row_overall = 0
        max_image_width_overall = 0 
        
        for subdir, images in sorted(all_images_by_subdir.items()):
            ws.cell(row=current_row, column=1, value=subdir).font = openpyxl.styles.Font(bold=True, size=14)
            current_col_for_images = 2
            max_height_in_row = 0
            
            if len(images) > max_graphs_in_a_row_overall:
                max_graphs_in_a_row_overall = len(images)

            for image_buffer in images:
                image_buffer.seek(0)
                img = OpenpyxlImage(image_buffer)
                
                if img.height > max_height_in_row:
                    max_height_in_row = img.height
                if img.width > max_image_width_overall:
                    max_image_width_overall = img.width
                
                col_letter = get_column_letter(current_col_for_images)
                ws.add_image(img, f'{col_letter}{current_row}')
                current_col_for_images += 1
            
            if max_height_in_row > 0:
                ws.row_dimensions[current_row].height = (max_height_in_row * 0.75) + SPACE_Y
            
            current_row += 2 
            
        if max_graphs_in_a_row_overall > 0:
            calculated_width = (max_image_width_overall * 0.135) + SPACE_X
            for i in range(2, max_graphs_in_a_row_overall + 2):
                ws.column_dimensions[get_column_letter(i)].width = calculated_width
                
        wb.save(output_excel_path)
        print(f"\n★ セル6完了: 全グラフのExcelファイルの保存が完了しました！ -> {os.path.basename(output_excel_path)}")


グラフの生成とExcelへの貼り付けを開始します（少し時間がかかります）...

--- サブフォルダ: 1.0mmNR\200℃ のグラフ生成 ---
グラフ作成中: 001_200℃_010_1.0mmNR.csv
グラフ作成中: 002_200℃_010_1.0mmNR.csv
グラフ作成中: 003_200℃_010_1.0mmNR.csv
グラフ作成中: 004_200℃_010_1.0mmNR.csv
グラフ作成中: 005_200℃_010_1.0mmNR.csv
グラフ作成中: 006_200℃_020_1.0mmNR.csv
グラフ作成中: 007_200℃_020_1.0mmNR.csv
グラフ作成中: 008_200℃_020_1.0mmNR.csv
グラフ作成中: 009_200℃_020_1.0mmNR.csv
グラフ作成中: 010_200℃_020_1.0mmNR.csv
グラフ作成中: 011_200℃_040_1.0mmNR.csv
グラフ作成中: 012_200℃_040_1.0mmNR.csv
グラフ作成中: 013_200℃_040_1.0mmNR.csv
グラフ作成中: 014_200℃_040_1.0mmNR.csv
グラフ作成中: 015_200℃_040_1.0mmNR.csv
グラフ作成中: 016_200℃_080_1.0mmNR.csv
グラフ作成中: 017_200℃_080_1.0mmNR.csv
グラフ作成中: 018_200℃_080_1.0mmNR.csv
グラフ作成中: 019_200℃_080_1.0mmNR.csv
グラフ作成中: 020_200℃_080_1.0mmNR.csv
グラフ作成中: 021_200℃_120_1.0mmNR.csv
グラフ作成中: 022_200℃_120_1.0mmNR.csv
グラフ作成中: 023_200℃_120_1.0mmNR.csv
グラフ作成中: 024_200℃_120_1.0mmNR.csv
グラフ作成中: 025_200℃_120_1.0mmNR.csv
グラフ作成中: 026_200℃_160_1.0mmNR.csv
グラフ作成中: 027_200℃_160_1.0mmNR.csv
グラフ作成中: 028_200℃_160_1.0mmNR.cs